<a href="https://colab.research.google.com/github/justamy20/scikit-learn-Cookbook/blob/main/Chapter_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 6: Regresi Logistik Lanjutan dan Ekstensinya

<div class="alert alert-info">
<b>Ringkasan Bab:</b> Bab ini memperdalam pemahaman tentang Logistic Regression. Kita akan membahas fungsi Sigmoid untuk klasifikasi biner, pendekatan Multiclass (One-vs-Rest dan Multinomial/Softmax), serta penggunaan parameter <code>C</code> untuk mengontrol regularisasi guna mencegah overfitting.
</div>

## 1. Regresi Logistik Dasar (Klasifikasi Biner)
Berbeda dengan regresi linear yang menghasilkan nilai kontinu (garis lurus), regresi logistik menggunakan **Fungsi Sigmoid** untuk memampatkan *output* menjadi nilai probabilitas antara 0 dan 1.

Fungsi Sigmoid dirumuskan sebagai:
$$p(X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X)}}$$

Jika $p(X) \geq 0.5$, data diprediksi masuk ke Kelas 1. Jika tidak, masuk ke Kelas 0.

In [16]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Membuat dataset biner (2 kelas)
X_bin, y_bin = make_classification(n_samples=200, n_features=4, n_classes=2, random_state=42)
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_bin, y_bin, test_size=0.3, random_state=42)

# Inisialisasi dan fit model
log_reg = LogisticRegression(solver='lbfgs')
log_reg.fit(X_train_b, y_train_b)

# Prediksi dan evaluasi
y_pred_b = log_reg.predict(X_test_b)
print(f"Akurasi Logistic Regression (Biner): {accuracy_score(y_test_b, y_pred_b) * 100:.2f}%")

# Melihat probabilitas dari 3 data pertama di test set
probabilitas = log_reg.predict_proba(X_test_b[:3])
print("\nProbabilitas [Kelas 0, Kelas 1] untuk 3 data pertama:")
print(np.round(probabilitas, 3))

Akurasi Logistic Regression (Biner): 93.33%

Probabilitas [Kelas 0, Kelas 1] untuk 3 data pertama:
[[0.458 0.542]
 [0.501 0.499]
 [0.083 0.917]]


---
## 2. Klasifikasi Multiclass (Banyak Kelas)
Bagaimana jika kita harus membedakan 3 kelas atau lebih (misalnya membedakan 3 jenis bunga)? Logistic Regression memiliki dua pendekatan utama:

1. **One-vs-Rest (OvR):** Membuat satu model biner untuk setiap kelas (misal: "Kelas A vs Bukan Kelas A").
2. **Multinomial (Softmax):** Menghitung probabilitas semua kelas secara bersamaan menggunakan fungsi Softmax.

Fungsi Softmax:
$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

In [17]:
from sklearn.datasets import load_iris

# Load dataset Iris (3 kelas bunga)
iris = load_iris()
X_multi, y_multi = iris.data, iris.target
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, test_size=0.3, random_state=42)

# Model 1: Pendekatan One-vs-Rest (OvR)
log_reg_ovr = LogisticRegression(multi_class='ovr', solver='liblinear')
log_reg_ovr.fit(X_train_m, y_train_m)

# Model 2: Pendekatan Multinomial (Softmax)
log_reg_multi = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=200)
log_reg_multi.fit(X_train_m, y_train_m)

print(f"Akurasi Multiclass (OvR): {log_reg_ovr.score(X_test_m, y_test_m) * 100:.2f}%")
print(f"Akurasi Multiclass (Multinomial): {log_reg_multi.score(X_test_m, y_test_m) * 100:.2f}%")

Akurasi Multiclass (OvR): 97.78%
Akurasi Multiclass (Multinomial): 100.00%


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


---
## 3. Regularisasi pada Logistic Regression (Parameter `C`)
Sama seperti Ridge dan Lasso, Logistic Regression di `scikit-learn` secara *default* sudah menerapkan regularisasi L2. Kekuatan regularisasi ini dikontrol oleh hyperparameter `C`.

<div class="alert alert-warning">
<b>Penting tentang Parameter C:</b><br>
Nilai <code>C</code> adalah <b>kebalikan</b> dari kekuatan regularisasi.<br>
- <code>C</code> yang sangat KECIL (misal: 0.01) = Regularisasi sangat KUAT (model lebih sederhana).<br>
- <code>C</code> yang sangat BESAR (misal: 100) = Regularisasi sangat LEMAH (model bisa sangat kompleks/overfit).
</div>

In [18]:
# Mari kita uji pengaruh parameter C pada dataset yang lebih rumit
X_c, y_c = make_classification(n_samples=500, n_features=20, n_informative=10, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.3, random_state=42)

# C = 0.01 (Regularisasi Kuat, mencekik bobot fitur agar mendekati 0)
log_reg_C_kecil = LogisticRegression(C=0.01, solver='lbfgs')
log_reg_C_kecil.fit(X_train_c, y_train_c)

# C = 100 (Regularisasi Lemah, membiarkan bobot fitur tumbuh bebas)
log_reg_C_besar = LogisticRegression(C=100, solver='lbfgs', max_iter=500)
log_reg_C_besar.fit(X_train_c, y_train_c)

print(f"Akurasi dengan C=0.01 (Train): {log_reg_C_kecil.score(X_train_c, y_train_c):.3f}")
print(f"Akurasi dengan C=0.01 (Test) : {log_reg_C_kecil.score(X_test_c, y_test_c):.3f}\n")

print(f"Akurasi dengan C=100 (Train) : {log_reg_C_besar.score(X_train_c, y_train_c):.3f}")
print(f"Akurasi dengan C=100 (Test)  : {log_reg_C_besar.score(X_test_c, y_test_c):.3f}")
print("\n*Perhatikan apakah C=100 mengalami overfitting (Train sangat tinggi, Test lebih rendah)*")

Akurasi dengan C=0.01 (Train): 0.826
Akurasi dengan C=0.01 (Test) : 0.847

Akurasi dengan C=100 (Train) : 0.837
Akurasi dengan C=100 (Test)  : 0.807

*Perhatikan apakah C=100 mengalami overfitting (Train sangat tinggi, Test lebih rendah)*
